In [29]:
import json
import pandas as pd
import re
from sklearn.model_selection import train_test_split

In [30]:
## Task 1: Clean & Split Data — Katie & Aarnav

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip -o "/content/cuad-v1.zip" -d "/content/cuad_data"

Archive:  /content/cuad-v1.zip
  inflating: /content/cuad_data/CUADv1.json  
  inflating: /content/cuad_data/test.json  
  inflating: /content/cuad_data/train_separate_questions.json  


In [32]:
!ls -R /content/cuad_data

/content/cuad_data:
CUADv1.json  test.json	train_separate_questions.json


In [33]:
# load the full CUAD dataset (all 510 contracts)
with open("/content/cuad_data/CUADv1.json", "r") as f:
    data = json.load(f)

print(f"Total contracts: {len(data['data'])}")

# we're building two separate tables from this one file:
# 1. contracts -- the contract name + its full text (for chunking later)
# 2. annotations -- every labeled clause span (for classification labels)
contracts = []
annotations = []

for contract in data["data"]:
    title = contract["title"]

    # each contract has one paragraph with a "context" field
    # that holds the entire contract text (can be 1,000 to 300,000+ chars)
    context = contract["paragraphs"][0]["context"]

    # one row per contract: just the name and full text
    contracts.append({"contract_title": title, "context": context})

    # now loop through the question-answer pairs for this contract
    # each qa asks "does category X appear in this contract?"
    # if yes, the answer contains the exact text the expert highlighted
    for qa in contract["paragraphs"][0]["qas"]:

        # the qa id looks like "ContractName__Parties_0"
        # split on "__" and take the last part to get the category name
        raw_category = qa["id"].split("__")[-1] if "__" in qa["id"] else qa.get("id", "Unknown")
        question = qa["question"]
        is_impossible = qa.get("is_impossible", False)

        # skip if no answers -- means this category doesn't exist in this contract
        if qa.get("answers"):
            for ans in qa["answers"]:
                annotations.append({
                    "contract_title": title,
                    "category_raw": raw_category,
                    "clause_text": ans["text"],          # the exact clause the expert highlighted
                    "answer_start": ans["answer_start"],  # where in the contract text it starts (char position)
                    "question": question,
                    "is_impossible": is_impossible
                })

# turn both lists into pandas tables
contracts_df = pd.DataFrame(contracts)
annotations_df = pd.DataFrame(annotations)

Total contracts: 510


In [34]:
# are any cells empty?
print("Missing values:")
print(annotations_df.isnull().sum())

# are any rows exact copies of another row?
print(f"\nDuplicate rows: {annotations_df.duplicated().sum()}")

# did any expert accidentally highlight nothing (blank text)?
blank = annotations_df[annotations_df["clause_text"].str.strip() == ""]
print(f"Blank clause text: {len(blank)}")

# did any expert accidentally highlight just 1-2 characters?
# these are bad labels -- things like "." tagged as a date
# or "GA" tagged as a party name
short = annotations_df[annotations_df["clause_text"].str.strip().str.len() <= 2]
print(f"Short answers (<=2 chars): {len(short)}")
if len(short) > 0:
    print(short[["contract_title", "category_raw", "clause_text"]])

# now actually fix the problems we found

# remember how many rows we started with
before = len(annotations_df)

# drop the rows where the clause text is 2 characters or less
annotations_df = annotations_df[annotations_df["clause_text"].str.strip().str.len() > 2]

# drop exact duplicate rows just in case
annotations_df = annotations_df.drop_duplicates().reset_index(drop=True)

# clean up extra spaces at the start/end of clause text
annotations_df["clause_text"] = annotations_df["clause_text"].str.strip()

# show how many rows we removed
print(f"\nAnnotations: {before} -> {len(annotations_df)} ({before - len(annotations_df)} removed)")






Missing values:
contract_title    0
category_raw      0
clause_text       0
answer_start      0
question          0
is_impossible     0
dtype: int64

Duplicate rows: 0
Blank clause text: 0
Short answers (<=2 chars): 17
                                          contract_title     category_raw  \
534    BONTONSTORESINC_04_20_2018-EX-99.3-AGENCY AGRE...          Parties   
3479   MIDDLEBROOKPHARMACEUTICALS,INC_03_18_2010-EX-1...          Parties   
4229   AMBASSADOREYEWEARGROUPINC_11_17_1997-EX-10.28-...          Parties   
4231   AMBASSADOREYEWEARGROUPINC_11_17_1997-EX-10.28-...          Parties   
4536   SUMMAFOURINC_06_19_1998-EX-10.3-SOFTWARE LICEN...          Parties   
4719   PhasebioPharmaceuticalsInc_20200330_10-K_EX-10...          Parties   
5561   ALLISONTRANSMISSIONHOLDINGSINC_12_15_2014-EX-9...   Effective Date   
8022   BloomEnergyCorp_20180321_DRSA (on S-1)_EX-10_1...          Parties   
8471   ROCKYMOUNTAINCHOCOLATEFACTORY,INC_12_23_2019-E...          Parties   
9179   Soup

In [35]:
# right now the categories have numbers at the end like Parties_0, Parties_1, Parties_2
# those numbers just mean "first time Parties appeared in this contract", "second time", etc.
# they're all the same category -- just different occurrences
# we need to remove those numbers so they all become just "Parties"
# without this, the model would think there are 408 categories instead of the correct 41

def get_base_category(cat):
    # looks for a pattern like _0, _1, _2, _10 at the very end of the string
    # and removes it, leaving just the category name
    return re.sub(r'_\d+$', '', cat)

# run that function on every row's category and save it as a new column called "category"
# we keep "category_raw" around in case anyone needs the original numbering later
annotations_df["category"] = annotations_df["category_raw"].apply(get_base_category)

# this should print 41 -- the CUAD dataset has exactly 41 clause categories
# if it prints anything else, something went wrong with the stripping
print(f"Unique categories: {annotations_df['category'].nunique()}")

# show how many examples each category has
# notice the imbalance: Parties has ~2000 examples while some categories have under 30
# Task 2 (Vrinda and Hadent) will visualize this in their EDA
print(annotations_df["category"].value_counts())

Unique categories: 41
category
Parties                               2539
License Grant                          777
Cap On Liability                       672
Anti-Assignment                        653
Audit Rights                           643
Insurance                              560
Document Name                          521
Agreement Date                         476
Expiration Date                        467
Governing Law                          464
Post-Termination Services              450
Effective Date                         446
Minimum Commitment                     424
Revenue/Profit Sharing                 418
Exclusivity                            410
Rofr/Rofo/Rofn                         367
Ip Ownership Assignment                318
Non-Transferable License               298
Non-Compete                            259
Change Of Control                      253
Termination For Convenience            246
Renewal Term                           210
Warranty Duration      

In [36]:
# the full contract text in contracts_df has formatting issues from the original PDF extraction
# things like: \r\n mixed with \n, multiple spaces in a row, huge gaps of blank lines
# we need to clean these up so the text is consistent
# BUT we do NOT remove legal terms, punctuation, or anything meaningful
# the github doc says "preserve important legal terminology during cleaning"

def clean_contract_text(text):
    # some lines end with \r\n (Windows style) and some with just \n (Mac/Linux style)
    # make them all \n so the text is consistent
    text = text.replace('\r\n', '\n').replace('\r', '\n')

    # collapse runs of multiple spaces or tabs into a single space
    # e.g. "the    Company" becomes "the Company"
    # this happens a lot in contracts converted from PDF
    text = re.sub(r'[ \t]+', ' ', text)

    # collapse 3+ blank lines in a row into just 2
    # contracts often have huge gaps between sections from the PDF layout
    # we keep double line breaks (they mark section boundaries) but remove excessive ones
    text = re.sub(r'\n{3,}', '\n\n', text)

    # remove any leading/trailing whitespace from the whole document
    return text.strip()

# apply that cleaning to every contract's full text
contracts_df["context"] = contracts_df["context"].apply(clean_contract_text)

# print some stats so we can sanity check the contracts look reasonable
# min should be around 1,000 chars (a very short contract)
# max should be around 300,000 chars (a very long one)
print(f"Contract lengths (chars):")
print(f"  Min: {contracts_df['context'].str.len().min()}")
print(f"  Max: {contracts_df['context'].str.len().max()}")
print(f"  Avg: {contracts_df['context'].str.len().mean():.0f}")

Contract lengths (chars):
  Min: 645
  Max: 300304
  Avg: 50632


In [37]:
# Standardize contract IDs, clause labels, annotation spans, and document metadata

print("--- Contract IDs Standardization ---")
# Contract IDs (titles) are used as unique identifiers across the dataset.
# The 'contract_title' column serves this purpose, extracted directly from source JSON.
# They are used as provided by the original dataset after loading.
print(f"Number of unique contract titles in contracts_df: {contracts_df['contract_title'].nunique()}")
print(f"Sample contract titles: {contracts_df['contract_title'].head(2).tolist()}")

print("\n--- Clause Labels Standardization ---")
# Clause labels were standardized by creating a 'category' column from 'category_raw',
# removing numerical suffixes. This ensures that 'Parties_0', 'Parties_1', etc.,
# all map to 'Parties'.

def get_base_category(cat):
    # looks for a pattern like _0, _1, _2, _10 at the very end of the string
    # and removes it, leaving just the category name
    return re.sub(r'_\d+$', '', cat)

# Ensure the 'category' column exists for demonstration
if 'category' not in annotations_df.columns:
    annotations_df["category"] = annotations_df["category_raw"].apply(get_base_category)

print(f"Number of unique raw categories: {annotations_df['category_raw'].nunique()}")
print(f"Number of unique standardized categories: {annotations_df['category'].nunique()}")
print("Top 5 standardized categories and their counts:")
print(annotations_df['category'].value_counts().head())

print("\n--- Annotation Spans Standardization ---")
# Annotation spans (clause_text) were cleaned in a previous step (cell uvZTjGG_TqIV)
# by removing short/blank entries and stripping leading/trailing whitespace.
# 'answer_start' is a numerical index, inherently standardized.
print(f"Length of annotations_df after cleaning clause_text: {len(annotations_df)}")
print(f"Sample cleaned clause text: '{annotations_df['clause_text'].iloc[0]}'"
)
print(f"Type of answer_start: {annotations_df['answer_start'].dtype}")

print("\n--- Document Metadata (Contract Text) Standardization ---")
# Document metadata (full contract text in 'context' column of contracts_df)
# was standardized in a previous step (cell xj2-4AxJl31E) by:
# - Normalizing line endings (\r\n to \n)
# - Collapsing multiple spaces/tabs into single spaces
# - Collapsing excessive blank lines (3+ into 2)
# This improves consistency without altering legal terminology.
print(f"Sample of cleaned contract text (first 200 chars):")
print(f"'{contracts_df['context'].iloc[0][:200]}...'"
)
print(f"Average contract text length after cleaning: {contracts_df['context'].str.len().mean():.0f} characters")

--- Contract IDs Standardization ---
Number of unique contract titles in contracts_df: 510
Sample contract titles: ['LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGREEMENT', 'WHITESMOKE,INC_11_08_2011-EX-10.26-PROMOTION AND DISTRIBUTION AGREEMENT']

--- Clause Labels Standardization ---
Number of unique raw categories: 41
Number of unique standardized categories: 41
Top 5 standardized categories and their counts:
category
Parties             2539
License Grant        777
Cap On Liability     672
Anti-Assignment      653
Audit Rights         643
Name: count, dtype: int64

--- Annotation Spans Standardization ---
Length of annotations_df after cleaning clause_text: 13806
Sample cleaned clause text: 'DISTRIBUTOR AGREEMENT'
Type of answer_start: int64

--- Document Metadata (Contract Text) Standardization ---
Sample of cleaned contract text (first 200 chars):
'EXHIBIT 10.6

 DISTRIBUTOR AGREEMENT

 THIS DISTRIBUTOR AGREEMENT (the "Agreement") is made by and between Electric City Corp., a Dela

In [38]:
# Load contract titles for the official test set
with open("/content/cuad_data/test.json", "r") as f:
    test_data = json.load(f)
test_contract_titles = [item["title"] for item in test_data["data"]]

# Load contract titles for the official training set
with open("/content/cuad_data/train_separate_questions.json", "r") as f:
    train_data = json.load(f)
train_contract_titles = [item["title"] for item in train_data["data"]]

# Create the official test set annotations DataFrame
test_annotations_df = annotations_df[annotations_df["contract_title"].isin(test_contract_titles)].copy()

# Create the combined training and validation set annotations DataFrame
train_val_annotations_df = annotations_df[annotations_df["contract_title"].isin(train_contract_titles)].copy()

print(f"Official Test Set Annotations: {len(test_annotations_df)}")
print(f"Combined Train/Validation Set Annotations: {len(train_val_annotations_df)}")

# Get unique contract titles from the combined train/validation set for a contract-wise split
unique_train_val_contracts = train_val_annotations_df["contract_title"].unique()

# Split the unique contract titles into training and validation subsets
train_contract_subset, val_contract_subset = train_test_split(
    unique_train_val_contracts, test_size=0.2, random_state=42
)

# Filter the combined train/validation annotations DataFrame based on these subsets
train_annotations_df = train_val_annotations_df[train_val_annotations_df["contract_title"].isin(train_contract_subset)].copy()
val_annotations_df = train_val_annotations_df[train_val_annotations_df["contract_title"].isin(val_contract_subset)].copy()

print(f"\nFinal Training Set Annotations: {len(train_annotations_df)}")
print(f"Validation Set Annotations: {len(val_annotations_df)}")

# Display the number of unique contracts in each set
print(f"\nNumber of unique contracts in Official Test Set: {test_annotations_df['contract_title'].nunique()}")
print(f"Number of unique contracts in Final Training Set: {train_annotations_df['contract_title'].nunique()}")
print(f"Number of unique contracts in Validation Set: {val_annotations_df['contract_title'].nunique()}")

# Also create corresponding splits for the contracts_df
test_contracts_df = contracts_df[contracts_df["contract_title"].isin(test_contract_titles)].copy()
train_contracts_df = contracts_df[contracts_df["contract_title"].isin(train_contract_subset)].copy()
val_contracts_df = contracts_df[contracts_df["contract_title"].isin(val_contract_subset)].copy()

print(f"\nNumber of contracts in Official Test Set (contracts_df): {len(test_contracts_df)}")
print(f"Number of contracts in Final Training Set (contracts_df): {len(train_contracts_df)}")
print(f"Number of contracts in Validation Set (contracts_df): {len(val_contracts_df)}")

Official Test Set Annotations: 2638
Combined Train/Validation Set Annotations: 11168

Final Training Set Annotations: 9165
Validation Set Annotations: 2003

Number of unique contracts in Official Test Set: 102
Number of unique contracts in Final Training Set: 326
Number of unique contracts in Validation Set: 82

Number of contracts in Official Test Set (contracts_df): 102
Number of contracts in Final Training Set (contracts_df): 326
Number of contracts in Validation Set (contracts_df): 82


In [39]:
# save the final files so teammates can load them directly
train_contracts_df.to_csv("train_contracts.csv", index=False)
val_contracts_df.to_csv("val_contracts.csv", index=False)
test_contracts_df.to_csv("test_contracts.csv", index=False)
train_annotations_df.to_csv("train_annotations.csv", index=False)
val_annotations_df.to_csv("val_annotations.csv", index=False)
test_annotations_df.to_csv("test_annotations.csv", index=False)

print("Saved 6 files:")
print(f"  train_contracts.csv  ({len(train_contracts_df)} contracts)")
print(f"  val_contracts.csv    ({len(val_contracts_df)} contracts)")
print(f"  test_contracts.csv   ({len(test_contracts_df)} contracts)")
print(f"  train_annotations.csv ({len(train_annotations_df)} annotations)")
print(f"  val_annotations.csv   ({len(val_annotations_df)} annotations)")
print(f"  test_annotations.csv  ({len(test_annotations_df)} annotations)")

Saved 6 files:
  train_contracts.csv  (326 contracts)
  val_contracts.csv    (82 contracts)
  test_contracts.csv   (102 contracts)
  train_annotations.csv (9165 annotations)
  val_annotations.csv   (2003 annotations)
  test_annotations.csv  (2638 annotations)


In [40]:
## Task 2: Run EDA  — Hadent & Vrinda

In [41]:
df_dict = {
    "train_contracts_df": train_contracts_df,
    "val_contracts_df": val_contracts_df,
    "test_contracts_df": test_contracts_df,
    "train_annotations_df": train_annotations_df,
    "val_annotations_df": val_annotations_df,
    "test_annotations_df": test_annotations_df
}

In [42]:
for df_name, df in df_dict.items():
    print(f"--- {df_name} ---")
    print(list(df.columns))
    print(f"-----------------\n")

--- train_contracts_df ---
['contract_title', 'context']
-----------------

--- val_contracts_df ---
['contract_title', 'context']
-----------------

--- test_contracts_df ---
['contract_title', 'context']
-----------------

--- train_annotations_df ---
['contract_title', 'category_raw', 'clause_text', 'answer_start', 'question', 'is_impossible', 'category']
-----------------

--- val_annotations_df ---
['contract_title', 'category_raw', 'clause_text', 'answer_start', 'question', 'is_impossible', 'category']
-----------------

--- test_annotations_df ---
['contract_title', 'category_raw', 'clause_text', 'answer_start', 'question', 'is_impossible', 'category']
-----------------



In [43]:
train

NameError: name 'train' is not defined